# Bayesian Analysis

**Author:** Monika Doerig, 23 Feb 2026  

[<img src="https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png" width="20"> MoniDoerig](https://github.com/MoniDoerig)

[<img src="https://info.orcid.org/wp-content/uploads/2019/11/orcid_16x16.png" width="20"> 0009-0008-3617-2185](https://orcid.org/0009-0008-3617-2185)

In [1]:
import module
await module.load('fsl/6.0.7.8')
await module.list()

['fsl/6.0.7.8']

In [2]:
%%capture
!pip install nilearn nibabel pandas pingouin

In [3]:
import os
import subprocess
import pandas as pd
import numpy as np
from nilearn import image, datasets
import nibabel as nib
import pingouin as pg

## 1. Bayesian Regression Analysis (Patient Group)

- Variable 1: Brain data (the extracted mean values from ROI for each of the 45 patients).
- Variable 2: Behavioral measure of interest (e.g., pain_duration for each of the 45 patients).

The analysis will produce a Bayes Factor (BF₁₀) that quantifies the evidence for the alternative hypothesis (H₁: there is a correlation) versus the null hypothesis (H₀: there is no correlation).

Your FSL randomise setup is doing a voxel-wise multiple regression. 
The Bayesian equivalent will be an ROI-based analysis.

- Use ROI BA areas 1-5
- Extract Data: For each patient, extract the mean value from the relevant brain map (e.g., PAT_45_Seg20minusRan20).
- Perform Bayesian Correlation: For each behavioral regressor (pain duration, TSK, etc.), run a separate Bayesian correlation test between the extracted brain data and that regressor.
- Report the BF₁₀: For each relationship, we'll get a BF₁₀ that tells us if there's evidence for a correlation or evidence for the absence of a correlation.

In [4]:
# --- Configuration ---

# Patient Data: files are already merged
base_path = '/mnt/neurodesktop-storage/Data/tempRSA/RSA'

# Dictionary of the two specific brain data files to be analyzed
brain_files_to_analyze = {
    "Seg80_minus_Ran80": os.path.join(base_path, 'Model_Comparison_PAT/input/N45_Seg80minusRan80.nii.gz'),
    "Seg20_minus_Ran20": os.path.join(base_path, 'Model_Comparison_PAT/input/N45_Seg20minusRan20.nii.gz')
}

# Path to the ROI mask we created in the previous step
#mask_path = os.path.join(os.getcwd(), 'combined_roi_mask.nii.gz')
mask_path = "/mnt/neurodesktop-storage/Siibra_masks/all_regs_1to5_BL_bin.nii.gz" # roi_mask


fremantle_pat = [18., 6, 5, 5, 3, 7, 4, 27, 11, 1, 1, 8, 7, 10, 3, 6, 3, 3, 6, 0, 10, 10, 4, 6, 2, 3, 3, 3, 3, 13, 5, 8, 8, 3, 6, 8, 3, 5, 15, 7, 22, 8, 15, 7, 14]
pain_duration = [562., 2300, 859, 5615, 2296, 2627 ,2194, 1205, 1527, 228, 4192, 7849, 2653, 1281, 775, 403, 4456, 2593, 1032, 7142, 5518, 4371, 4715, 1435, 1945, 3275, 
                 2599, 1842, 1473, 3275, 1838, 4279, 1493, 150, 2278, 3739, 3359, 2673, 4476, 2292, 2671, 5069, 2337, 2214, 4272]
pain_mri =[1., 2, 4, 3, 2, 0, 3, 4, 1, 3, 3, 1, 0, 6, 0, 3, 1, 1, 2, 0, 4, 4, 6, 3, 2, 3, 3, 3, 1, 2, 2, 3, 3, 0, 3, 0, 3, 2, 3, 3, 5, 0, 6, 0, 2]
avg_pain_4weeks = [4., 5, 5, 3, 2, 6, 1, 5, 3, 6, 4, 4, 4, 4, 2, 7, 1, 4, 3, 3, 3, 5, 6, 4, 1, 3, 5, 4, 2, 2, 3, 4, 5, 2, 4, 1, 5, 7, 6, 6, 4, 4, 4, 1, 3] 
max_pain = [4., 8, 8, 6, 5, 7, 4, 10, 5, 10, 5, 6, 5, 7, 4, 8, 3, 6, 7, 7, 5, 7, 9, 7, 1, 5, 8, 9, 6, 6, 5, 7, 6, 5, 6, 5, 6, 9, 8, 8, 7, 6, 7, 4, 7]
tsk = [35., 47, 39, 27, 30, 31, 36, 35, 34, 43, 33, 28, 28, 35, 30, 39, 38, 30, 29, 27, 33, 34, 35, 22, 30, 23, 37, 25, 30, 34, 26, 34, 31, 38, 35, 30, 49, 31, 28, 38, 29, 25, 26, 37, 24]
pain_detect = [23., 13, 25, 12, 13, 18, 13, 25, 14, 12, 9, 18, 13, 15, 14, 20, 11, 13, 19, 9, 11, 20, 20, 15, 10, 9, 11, 16, 16, 12, 9, 11, 14, 15, 14, 16, 15, 14, 17, 4, 18, 18, 15, 4, 10]


# Store regressors in a dictionary for easy iteration
regressors = {
    "pain_duration": pain_duration,
    "fremantle": fremantle_pat,
    "pain_mri": pain_mri,
    "avg_pain_4weeks": avg_pain_4weeks,
    "max_pain_4weeks": max_pain,
    "tsk": tsk,
    "pain_detect": pain_detect
}

In [5]:
# --- Main Analysis Loop ---
all_results = []
print("--- Starting Focused Bayesian Regression Analysis ---")

# Loop through each of the two specified brain data files
for model_name, brain_file_path in brain_files_to_analyze.items():
    print(f"\nProcessing Model Contrast: {model_name}")
    
    if not os.path.exists(brain_file_path):
        print(f"   -> WARNING: File not found, skipping: {brain_file_path}")
        continue

    # --- 1. Extract Brain Data from ROI ---
    output_txt_path = os.path.join(os.getcwd(), f'extracted_reg_values_{model_name}.txt')
    command = ['fslmeants', '-i', brain_file_path, '-m', mask_path, '-o', output_txt_path]
    
    try:
        subprocess.run(command, check=True, capture_output=True, text=True)
        brain_values = np.loadtxt(output_txt_path)
        print(f"   -> Data extracted successfully for {len(brain_values)} subjects.")
    except FileNotFoundError:
        print("   -> ERROR: 'fslmeants' not found. Ensure FSL is installed and in your system's PATH.")
        break
    except subprocess.CalledProcessError as e:
        print(f"   -> ERROR running fslmeants for {model_name}:\n{e.stderr}")
        continue

    # --- Perform Bayesian Correlation for Each Regressor ---
    for regressor_name, regressor_data in regressors.items():
        df = pd.DataFrame({'BrainData': brain_values, 'BehavioralData': regressor_data})
        corr_stats = pg.corr(df['BrainData'], df['BehavioralData'], method="pearson")
        
        all_results.append({
            'Model Contrast': model_name,
            'Regressor': regressor_name,
            'Pearson r': corr_stats['r'].iloc[0],
            'p-value': corr_stats['p_val'].iloc[0],
            'Bayes Factor (BF10)': corr_stats['BF10'].iloc[0]
        })

--- Starting Focused Bayesian Regression Analysis ---

Processing Model Contrast: Seg80_minus_Ran80
   -> Data extracted successfully for 45 subjects.

Processing Model Contrast: Seg20_minus_Ran20
   -> Data extracted successfully for 45 subjects.


In [6]:
# --- Display Final Consolidated Results ---
print("\n\n--- Consolidated Bayesian Regression Analysis Results ---")

if not all_results:
    print("No results were generated. Please check file paths and FSL installation.")
else:
    final_df = pd.DataFrame(all_results)
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', 120)
    print(final_df)

print("\n--- Interpretation Guide ---")
print("BF10 > 3:    Substantial evidence FOR a correlation (H1)")
print("BF10 < 1/3:  Substantial evidence for NO correlation (H0)")
print("1/3 < BF10 < 3: Anecdotal evidence (inconclusive)")
print("\nAnalysis complete.")



--- Consolidated Bayesian Regression Analysis Results ---
       Model Contrast        Regressor  Pearson r   p-value Bayes Factor (BF10)
0   Seg80_minus_Ran80    pain_duration   0.082758  0.588876               0.214
1   Seg80_minus_Ran80        fremantle   0.054132  0.723963               0.197
2   Seg80_minus_Ran80         pain_mri  -0.045885  0.764711               0.194
3   Seg80_minus_Ran80  avg_pain_4weeks   0.300364  0.044989               1.302
4   Seg80_minus_Ran80  max_pain_4weeks   0.309077  0.038839               1.469
5   Seg80_minus_Ran80              tsk   0.183311  0.228077               0.376
6   Seg80_minus_Ran80      pain_detect   0.317993  0.033276               1.669
7   Seg20_minus_Ran20    pain_duration   0.145358  0.340730               0.288
8   Seg20_minus_Ran20        fremantle   0.015230  0.920902               0.187
9   Seg20_minus_Ran20         pain_mri   0.188659  0.214558               0.392
10  Seg20_minus_Ran20  avg_pain_4weeks   0.115229  0.451004 

## 2. Bayesian Group Comparison (HC/ LBP)

In [7]:
# --- Configuration ---
base_path = '/mnt/neurodesktop-storage/Data/tempRSA/RSA'

# Define your input files
input_files = {
    "Seg_20": os.path.join(base_path, 'group_diffs_HC_PAT/all_files_41HCs_45PAT_Seg20only.nii.gz'),
    "Seg_80": os.path.join(base_path, 'group_diffs_HC_PAT/all_files_41HCs_45PAT_Seg80only.nii.gz'),
    "Sim_20": os.path.join(base_path, 'group_diffs_HC_PAT/all_files_41HCs_45PAT_Sim20only.nii.gz'),
    "Sim_80": os.path.join(base_path, 'group_diffs_HC_PAT/all_files_41HCs_45PAT_Sim80only.nii.gz'),
    "Seg80_minus_Ran80": os.path.join(base_path, 'group_diffs_HC_PAT/all_files_41HCs_45PAT_Seg80minusRan80.nii.gz'),
    "Seg80_minus_Sim80": os.path.join(base_path, 'group_diffs_HC_PAT/all_files_41HCs_45PAT_Seg80minusSim80.nii.gz'),
    "Seg20_minus_Ran20": os.path.join(base_path, 'group_diffs_HC_PAT/all_files_41HCs_45PAT_Seg20minusRan20.nii.gz'),
    "Seg20_minus_Sim20": os.path.join(base_path, 'group_diffs_HC_PAT/all_files_41HCs_45PAT_Seg20minusSim20.nii.gz')
}

# Group sizes
n_hc = 41
n_pat = 45

In [8]:
# --- Extract Data using fslmeants ---
print("Extracting data for each input file using fslmeants...")
extracted_data_files = {}

for name, filepath in input_files.items():
    if not os.path.exists(filepath):
        print(f"   WARNING: File not found, skipping: {filepath}")
        continue
        
    output_txt_path = os.path.join(os.getcwd(), f'extracted_values_{name}.txt')
    
    # Build and execute the fslmeants command
    command = [
        'fslmeants',
        '-i', filepath,
        '-m', mask_path,
        '-o', output_txt_path
    ]
    
    print(f"   -> Running command for: {name}")
    try:
        # The subprocess.run command executes the shell command
        subprocess.run(command, check=True, capture_output=True, text=True)
        extracted_data_files[name] = output_txt_path
    except FileNotFoundError:
        print("   ERROR: 'fslmeants' was not found. Please ensure FSL is installed and its 'bin' directory is in your system's PATH.")
        exit()
    except subprocess.CalledProcessError as e:
        print(f"   ERROR running fslmeants for {name}:")
        print(e.stderr) # Print the error message from FSL
        continue

print("   -> Data extraction complete.\n")

Extracting data for each input file using fslmeants...
   -> Running command for: Seg_20
   -> Running command for: Seg_80
   -> Running command for: Sim_20
   -> Running command for: Sim_80
   -> Running command for: Seg80_minus_Ran80
   -> Running command for: Seg80_minus_Sim80
   -> Running command for: Seg20_minus_Ran20
   -> Running command for: Seg20_minus_Sim20
   -> Data extraction complete.



In [9]:
# --- Perform Bayesian Analysis with Pingouin ---
print("Running Bayesian t-tests...")
results_list = []

for name, txt_path in extracted_data_files.items():
    # Load the extracted values from the text file
    all_values = np.loadtxt(txt_path)
    
    # Split the data into the two groups (Healthy Controls and Patients)
    hc_values = all_values[:n_hc]
    pat_values = all_values[n_hc:]
    
    # Perform the t-test (this function calculates both frequentist and Bayesian stats)
    stats = pg.ttest(hc_values, pat_values, paired=False, correction='auto')
    
    # Extract the relevant statistics from the results dataframe
    t_stat = stats['T'].iloc[0]
    p_val = stats['p_val'].iloc[0]
    bf10 = stats['BF10'].iloc[0]
    
    results_list.append({
        'Contrast': name,
        'T-Statistic': t_stat,
        'p-value': p_val,
        'Bayes Factor (BF10)': bf10
    })

Running Bayesian t-tests...


In [10]:
# Create a pandas DataFrame for clean output
results_df = pd.DataFrame(results_list)

# --- Display Results ---
print("\n--- Bayesian Analysis Results ---\n")
# Use to_string() to ensure the full table is printed
print(results_df.to_string(index=False))

print("\n--- Interpretation of Bayes Factors ---")
print("BF10 > 3:    Substantial evidence for a difference (H1)")
print("BF10 < 1/3:  Substantial evidence for NO difference (H0)")
print("1/3 < BF10 < 3: Anecdotal evidence (inconclusive)")
print("\nAnalysis complete.")


--- Bayesian Analysis Results ---

         Contrast  T-Statistic  p-value Bayes Factor (BF10)
           Seg_20     0.653264 0.515457               0.272
           Seg_80     1.377859 0.172417               0.516
           Sim_20    -1.889426 0.062508               1.059
           Sim_80     0.972773 0.333681               0.341
Seg80_minus_Ran80     1.638424 0.105611               0.724
Seg80_minus_Sim80     0.921852 0.359250               0.327
Seg20_minus_Ran20     0.947176 0.346391               0.334
Seg20_minus_Sim20     2.566596 0.012057               3.799

--- Interpretation of Bayes Factors ---
BF10 > 3:    Substantial evidence for a difference (H1)
BF10 < 1/3:  Substantial evidence for NO difference (H0)
1/3 < BF10 < 3: Anecdotal evidence (inconclusive)

Analysis complete.


In [11]:
%load_ext watermark

%watermark
%watermark --iversions

Last updated: 2026-03-04T01:22:57.841804+00:00

Python implementation: CPython
Python version       : 3.13.9
IPython version      : 9.7.0

Compiler    : GCC 14.3.0
OS          : Linux
Release     : 5.15.0-164-generic
Machine     : x86_64
Processor   : x86_64
CPU cores   : 32
Architecture: 64bit

nibabel : 5.3.2
nilearn : 0.11.1
numpy   : 2.2.4
pandas  : 2.2.3
pingouin: 0.6.0

